# Human Action Recognition — Transformer Training

Replaces the LSTM notebook (`lstm_train.ipynb`) with a Spatiotemporal Transformer.

**Architecture summary**
```
Input (B, 32, 36)
  └─ Linear 36→128          patch / frame embedding
  └─ Sinusoidal PositionalEncoding
  └─ 4 × TransformerEncoderLayer (d=128, heads=8, ffn=256, drop=0.1)
  └─ Global Average Pool over T
  └─ Linear 128→64 → ReLU → Linear 64→6
```
6 classes: JUMPING, JUMPING_JACKS, BOXING, WAVING_2HANDS, WAVING_1HAND, CLAPPING_HANDS


## 1 · Environment setup

In [ ]:
# Run once in Colab
!pip install pytorch-lightning einops --quiet

## 2 · Download & unzip dataset

In [ ]:
!wget -O RNN-HAR-2D-Pose-database.zip \
    'https://drive.google.com/u/1/uc?id=1IuZlyNjg6DMQE3iaO1Px6h1yLKgatynt'
!unzip -q RNN-HAR-2D-Pose-database.zip

## 3 · Clone repo & install deps

In [ ]:
!git clone https://github.com/shsarv/Huam-Activity-Detection
%cd Huam-Activity-Detection/
!pip install -r requirements.txt --quiet

## 4 · Copy the transformer module into src/

In [ ]:
# Paste the full contents of src/transformer.py here, or upload the file.
# If you already placed transformer.py in the repo's src/ folder, skip this cell.
import os, textwrap
os.makedirs('src', exist_ok=True)

TRANSFORMER_CODE = '''
# ── src/transformer.py ───────────────────────────────────────────────────────
# (copy the full src/transformer.py content here if not already present)
'''

# Preferred: just upload src/transformer.py and skip this cell.

## 5 · Config

In [ ]:
from argparse import ArgumentParser

DATASET_PATH = "/content/RNN-HAR-2D-Pose-database/"  # adjust if needed

HPARAMS = dict(
    batch_size      = 256,
    epochs          = 50,
    learning_rate   = 1e-4,
    num_classes     = 6,
    d_model         = 128,
    nhead           = 8,
    num_layers      = 4,
    dim_feedforward = 256,
    dropout         = 0.1,
    input_size      = 36,   # 18 keypoints × 2
)
print(HPARAMS)

## 6 · Import model & data module

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor

# NEW import — transformer replaces lstm
from src.transformer import ActionClassificationTransformer, PoseDataModule

## 7 · Build model + data module

In [ ]:
pl.seed_everything(42)

model = ActionClassificationTransformer(
    input_size      = HPARAMS['input_size'],
    d_model         = HPARAMS['d_model'],
    nhead           = HPARAMS['nhead'],
    num_layers      = HPARAMS['num_layers'],
    dim_feedforward = HPARAMS['dim_feedforward'],
    dropout         = HPARAMS['dropout'],
    num_classes     = HPARAMS['num_classes'],
    learning_rate   = HPARAMS['learning_rate'],
)

data_module = PoseDataModule(
    data_root  = DATASET_PATH,
    batch_size = HPARAMS['batch_size'],
)

print(model)

## 8 · Callbacks

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor   = 'val_loss',
    save_top_k= 1,
    filename  = 'transformer-{epoch:02d}-{val_loss:.4f}',
    verbose   = True,
)

early_stop = EarlyStopping(
    monitor  = 'val_loss',
    patience = 10,
    verbose  = True,
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

## 9 · TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=lightning_logs

## 10 · Train

In [ ]:
trainer = pl.Trainer(
    max_epochs               = HPARAMS['epochs'],
    gpus                     = 1,          # set 0 for CPU
    deterministic            = True,
    progress_bar_refresh_rate= 5,
    callbacks                = [early_stop, checkpoint_callback, lr_monitor],
)

trainer.fit(model, data_module)

## 11 · Copy best checkpoint to models/saved_model.ckpt

The Flask app expects the checkpoint at exactly `models/saved_model.ckpt`.

In [ ]:
import os, shutil

def get_best_ckpt(lightning_logs_dir='lightning_logs'):
    best_ver = max(
        (int(d.split('_')[1]) for d in os.listdir(lightning_logs_dir) if 'version' in d),
        default=0
    )
    ckpt_dir = os.path.join(lightning_logs_dir, f'version_{best_ver}', 'checkpoints')
    ckpts    = [f for f in os.listdir(ckpt_dir) if f.endswith('.ckpt')]
    return os.path.join(ckpt_dir, ckpts[-1])

best = get_best_ckpt()
print('Best checkpoint:', best)

os.makedirs('models', exist_ok=True)
shutil.copy(best, 'models/saved_model.ckpt')
print('Saved → models/saved_model.ckpt')

## 12 · Quick evaluation

In [ ]:
import torch
import numpy as np
from src.transformer import ActionClassificationTransformer, PoseDataModule, LABELS

loaded = ActionClassificationTransformer.load_from_checkpoint('models/saved_model.ckpt')
loaded.eval()

dm = PoseDataModule(data_root=DATASET_PATH, batch_size=512)
dm.setup()

all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in dm.val_dataloader():
        preds = loaded(X).argmax(dim=1)
        all_preds.append(preds.numpy())
        all_labels.append(y.numpy())

preds  = np.concatenate(all_preds)
labels = np.concatenate(all_labels)
acc    = (preds == labels).mean()
print(f'Test accuracy: {acc:.4f}')

from sklearn.metrics import classification_report
print(classification_report(labels, preds, target_names=LABELS))